# Model Analysis: Transformer + Nystrom OC-SVM

Diagnostic analysis of the trained Transformer + Nystrom OC-SVM detector:
1. Inspect the internal OC-SVM state (gamma, rho, landmarks).
2. Compute latent-space statistics from the Transformer encoder.
3. Kernel sanity check: verify whether the RBF kernel is well-calibrated.
4. Decision function distribution on test data.

In [1]:
import os, sys, glob, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import joblib

sys.path.insert(0, os.path.abspath(".."))

from detection.data.loaders import create_sequences as _create_sequences, load_processed
from detection.models.transformer import BottleneckTransformer

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


## Configuration

In [5]:
DATA_DIR = os.path.join("..", "data", "processed", "TOTF.PA-book")
RESULTS_DIR = os.path.join("..", "results")

SEQ_LENGTH = 25

# LOB columns present in processed files (not features)
LOB_COLUMNS = [
    f"{side}-{typ}-{lvl}"
    for lvl in range(1, 11)
    for side, typ in [("bid","price"),("bid","volume"),("ask","price"),("ask","volume")]
]

FILES = sorted(glob.glob(os.path.join(DATA_DIR, "*.parquet")))
TEST_FILES = [FILES[22], FILES[23], FILES[24], FILES[25], FILES[26]]

## Load Feature Names

In [3]:
feat_path = os.path.join(RESULTS_DIR, "transformer_ocsvm_features.txt")
if os.path.exists(feat_path):
    with open(feat_path) as f:
        feat_names = [line.strip() for line in f if line.strip()]
else:
    _, _feat_tmp = load_processed(TEST_FILES[0], "xltime", LOB_COLUMNS)
    feat_names = _feat_tmp.columns.tolist()
    del _feat_tmp

num_features = len(feat_names)
print(f"Features: {num_features}")

Features: 89


## Nystrom OC-SVM Diagnostic

Load the trained Transformer encoder and Nystrom OC-SVM detector, then inspect:
- Internal OC-SVM parameters (gamma, rho, weight norm, landmarks).
- Latent-space statistics from the encoder.
- Whether the RBF kernel is well-calibrated for the latent representations.
- The decision function distribution on test data.

In [ ]:
weights_path = os.path.join(RESULTS_DIR, "transformer_ocsvm_weights.pth")

transformer = BottleneckTransformer(
    num_features=num_features, model_dim=128, num_heads=8,
    num_layers=6, representation_dim=128, sequence_length=SEQ_LENGTH,
    dim_feedforward=512,
)
transformer.load_state_dict(torch.load(weights_path, map_location=DEVICE, weights_only=True))
transformer.eval().to(DEVICE)

ocsvm_path = os.path.join(RESULTS_DIR, "transformer_ocsvm_detector.pth")
ocsvm = torch.load(ocsvm_path, map_location=DEVICE, weights_only=False)

print("=== Nystrom OC-SVM internal state ===")
print(f"  gamma      : {ocsvm._gamma}")
print(f"  rho        : {ocsvm._rho.item():.6f}")
print(f"  ||w||      : {ocsvm._w.norm().item():.6f}")
print(f"  n_landmarks: {ocsvm._landmarks.shape}")
print()

_, features_test = load_processed(TEST_FILES[0], "xltime", LOB_COLUMNS)
feat_df = features_test.copy()
for col in feat_names:
    if col not in feat_df.columns:
        feat_df[col] = 0.0
feat_df = feat_df[feat_names]
scaler_path = os.path.join(RESULTS_DIR, "transformer_ocsvm_scaler.pkl")
scaler = joblib.load(scaler_path)
scaled = scaler.transform(feat_df.values[:5000].astype(np.float32)).astype(np.float32)
seqs = _create_sequences(scaled, SEQ_LENGTH)
x_t = torch.tensor(seqs[:512], dtype=torch.float32).to(DEVICE)

with torch.no_grad():
    z = transformer.get_representation(x_t)

print("=== Latent space statistics (512 samples) ===")
print(f"  shape        : {z.shape}")
print(f"  mean         : {z.mean().item():.6f}")
print(f"  std          : {z.std().item():.6f}")
print(f"  var (overall): {z.var().item():.6f}")
print()

d = z.shape[1]
var_z = z.var().item()
expected_dist2 = 2 * d * var_z
gamma = ocsvm._gamma
typical_kernel = np.exp(-gamma * expected_dist2)
print("=== Kernel sanity check ===")
print(f"  d (latent dim)           : {d}")
print(f"  Var(z)                   : {var_z:.6f}")
print(f"  gamma                    : {gamma:.8f}")
print(f"  Expected ||x-y||^2       : {expected_dist2:.2f}")
print(f"  Typical K(x,y)           : {typical_kernel:.2e}")
print(f"  If K(x,y) ~ 0 for all pairs, Nystrom features collapse to ~0")
print(f"  and decision_function ~ -rho = {-ocsvm._rho.item():.6f} for ALL samples.")
print()

del features_test

=== Nystrom OC-SVM internal state ===
  gamma      : 1.1521486041269603e-05
  rho        : -0.369177
  ||w||      : 0.450679
  n_landmarks: torch.Size([300, 128])

=== Latent space statistics (512 samples) ===
  shape        : torch.Size([512, 128])
  mean         : -0.255681
  std          : 18.352898
  var (overall): 336.828827

=== Kernel sanity check ===
  d (latent dim)           : 128
  Var(z)                   : 336.828827
  gamma                    : 0.00001152
  Expected ||x-y||^2       : 86228.18
  Typical K(x,y)           : 3.70e-01
  If K(x,y) ~ 0 for all pairs, Nystrom features collapse to ~0
  and decision_function ~ -rho = 0.369177 for ALL samples.

